In [ ]:
"""
Merge IMD Table 3 (Synoptic Hourly) data for Mumbai Santacruz (43003) and Colaba (43057)
from 1969 to 2025, and convert HR codes to proper datetime.

Input files:
  - 43003_1969-2009_.csv          (Table 3, 1969-2009)
  - 43003_Table_3_Daily_NDCQ-2025-02-237.csv  (Table 3, 2010-2024)
  - 43003_2000-2025_.csv          (Table 3, 2020-2025)
  - 43057_1969-2009_.csv          (Table 3, 1969-2009)
  - 43057_Table_3_Daily_NDCQ-2025-02-237.csv  (Table 3, 2010-2024)
  - 43057_2000-2025_.csv          (Table 3, 2020-2025)

Output:
  - 43003_Table3_merged_1969_2025.csv
  - 43057_Table3_merged_1969_2025.csv

HR Code to IST mapping:
  00 → 05:30 IST    48 → 17:30 IST
  12 → 08:30 IST    60 → 20:30 IST
  24 → 11:30 IST    72 → 23:30 IST
  36 → 14:30 IST    84 → 02:30 IST (+1 day)
"""

import pandas as pd
import numpy as np
import os
from datetime import timedelta

# ============================================================
# CONFIGURATION
# ============================================================
UPLOAD_DIR = "/content/drive/MyDrive/Major_project_imd"
OUTPUT_DIR = "/content/drive/MyDrive/Major_project_imd"

# Files for each station
FILES = {
    "43003": {
        "1969_2009": os.path.join(UPLOAD_DIR, "43003(1969-2009).csv"),
        "2010_2024": os.path.join(UPLOAD_DIR, "43003_Table_3_Daily_NDCQ-2025-02-237.csv"),
        "2020_2025": os.path.join(UPLOAD_DIR, "43003(2000-2025).csv"),
    },
    "43057": {
        "1969_2009": os.path.join(UPLOAD_DIR, "43057(1969-2009).csv"),
        "2010_2024": os.path.join(UPLOAD_DIR, "43057_Table_3_Daily_NDCQ-2025-02-237.csv"),
        "2020_2025": os.path.join(UPLOAD_DIR, "43057(2000-2025).csv"),
    },
}

# HR code → (IST hour, IST minute, day_offset)
# day_offset = 1 for HR=84 because 02:30 IST is the next day
HR_TO_IST = {
    0:  (5,  30, 0),
    12: (8,  30, 0),
    24: (11, 30, 0),
    36: (14, 30, 0),
    48: (17, 30, 0),
    60: (20, 30, 0),
    72: (23, 30, 0),
    84: (2,  30, 1),  # next day
}


# ============================================================
# FUNCTIONS
# ============================================================

def load_csv(filepath):
    """Load a Table 3 CSV file with proper handling of whitespace and missing values."""
    df = pd.read_csv(
        filepath,
        skipinitialspace=True,
        na_values=["", " ", "  ", "   ", "    ", "/"],
    )
    # Strip whitespace from column names
    df.columns = df.columns.str.strip()

    # Convert key columns to numeric
    for col in ["YEAR", "MN", "HR", "DT"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    return df


def merge_station_files(station_id, file_dict):
    """
    Merge 3 files for a station:
      - 1969-2009 file → use for years 1969-2009
      - 2010-2024 file → use for years 2010-2019
      - For 2020-2025 → combine 2010-2024 and 2020-2025 files, deduplicate
    """
    print(f"\n{'='*60}")
    print(f"  Processing Station: {station_id}")
    print(f"{'='*60}")

    # --- Load all 3 files ---
    print(f"\n  Loading 1969-2009 file...")
    df_1969 = load_csv(file_dict["1969_2009"])
    print(f"    Rows: {len(df_1969)}, Years: {int(df_1969['YEAR'].min())}-{int(df_1969['YEAR'].max())}")

    print(f"  Loading 2010-2024 file...")
    df_2010 = load_csv(file_dict["2010_2024"])
    print(f"    Rows: {len(df_2010)}, Years: {int(df_2010['YEAR'].min())}-{int(df_2010['YEAR'].max())}")

    print(f"  Loading 2020-2025 file...")
    df_2020 = load_csv(file_dict["2020_2025"])
    print(f"    Rows: {len(df_2020)}, Years: {int(df_2020['YEAR'].min())}-{int(df_2020['YEAR'].max())}")

    # --- Split by year ranges ---
    # Part 1: 1969-2009 from first file
    part1 = df_1969[df_1969["YEAR"] <= 2009].copy()
    print(f"\n  Part 1 (1969-2009): {len(part1)} rows")

    # Part 2: 2010-2019 from second file only
    part2 = df_2010[(df_2010["YEAR"] >= 2010) & (df_2010["YEAR"] <= 2019)].copy()
    print(f"  Part 2 (2010-2019): {len(part2)} rows")

    # Part 3: 2020-2025 → combine both sources, then deduplicate
    part3a = df_2010[df_2010["YEAR"] >= 2020].copy()
    part3b = df_2020.copy()
    part3 = pd.concat([part3a, part3b], ignore_index=True)
    print(f"  Part 3 (2020-2025) before dedup: {len(part3)} rows")

    # Deduplicate on key columns (keep first occurrence)
    key_cols = ["INDEX", "YEAR", "MN", "HR", "DT"]
    part3 = part3.drop_duplicates(subset=key_cols, keep="first")
    print(f"  Part 3 (2020-2025) after dedup:  {len(part3)} rows")

    # --- Combine all parts ---
    merged = pd.concat([part1, part2, part3], ignore_index=True)
    print(f"\n  Total merged rows: {len(merged)}")

    return merged


def convert_datetime(df):
    """
    Convert YEAR, MN, DT, HR columns into a proper DATETIME_IST column.

    HR code mapping to IST:
        00 → 05:30    48 → 17:30
        12 → 08:30    60 → 20:30
        24 → 11:30    72 → 23:30
        36 → 14:30    84 → 02:30 (next day)
    """
    print(f"\n  Converting HR codes to IST datetime...")

    # Create base date from YEAR, MN, DT
    df["_base_date"] = pd.to_datetime(
        df[["YEAR", "MN", "DT"]].rename(
            columns={"YEAR": "year", "MN": "month", "DT": "day"}
        ),
        errors="coerce",
    )

    # Map HR code to IST hour, minute, and day offset
    df["_hr_int"] = df["HR"].astype(int)
    df["_ist_hour"] = df["_hr_int"].map(lambda x: HR_TO_IST.get(x, (None, None, None))[0])
    df["_ist_min"]  = df["_hr_int"].map(lambda x: HR_TO_IST.get(x, (None, None, None))[1])
    df["_day_off"]  = df["_hr_int"].map(lambda x: HR_TO_IST.get(x, (None, None, None))[2])

    # Flag rows with invalid HR codes
    invalid_hr = df["_ist_hour"].isna()
    if invalid_hr.any():
        print(f"    WARNING: {invalid_hr.sum()} rows with unrecognized HR codes — these will have NaT datetime")

    # Construct DATETIME_IST
    df["DATETIME_IST"] = pd.NaT
    valid = ~invalid_hr & df["_base_date"].notna()

    df.loc[valid, "DATETIME_IST"] = (
        df.loc[valid, "_base_date"]
        + pd.to_timedelta(df.loc[valid, "_ist_hour"].astype(int), unit="h")
        + pd.to_timedelta(df.loc[valid, "_ist_min"].astype(int), unit="m")
        + pd.to_timedelta(df.loc[valid, "_day_off"].astype(int), unit="D")
    )

    # Also add a UTC datetime column for reference
    df["DATETIME_UTC"] = df["DATETIME_IST"] - timedelta(hours=5, minutes=30)

    # Clean up temp columns
    df.drop(columns=["_base_date", "_hr_int", "_ist_hour", "_ist_min", "_day_off"], inplace=True)

    # Sort by datetime
    df.sort_values("DATETIME_IST", inplace=True)
    df.reset_index(drop=True, inplace=True)

    print(f"    Date range: {df['DATETIME_IST'].min()} to {df['DATETIME_IST'].max()}")
    print(f"    Valid datetimes: {df['DATETIME_IST'].notna().sum()} / {len(df)}")

    return df


def print_summary(df, station_id):
    """Print a summary report of the merged data."""
    print(f"\n  {'─'*50}")
    print(f"  SUMMARY FOR STATION {station_id}")
    print(f"  {'─'*50}")

    # Records per year
    yearly = df.groupby("YEAR").size()
    print(f"\n  Records per year:")
    for yr, cnt in yearly.items():
        flag = " ⚠️ LOW" if cnt < 300 else ""
        print(f"    {int(yr)}: {cnt:>5} records{flag}")

    # Key parameter availability
    print(f"\n  Key parameter availability (non-null %):")
    key_params = ["DBT", "WBT", "DPT", "RH", "VP", "FFF", "RF"]
    for param in key_params:
        if param in df.columns:
            # Convert to numeric first
            col = pd.to_numeric(df[param], errors="coerce")
            pct = col.notna().sum() / len(df) * 100
            print(f"    {param:>5}: {pct:6.1f}%")

    # HR code distribution
    print(f"\n  Observations by HR code:")
    hr_dist = df["HR"].value_counts().sort_index()
    for hr, cnt in hr_dist.items():
        ist_info = HR_TO_IST.get(int(hr), None)
        if ist_info:
            h, m, _ = ist_info
            print(f"    HR {int(hr):>2} ({h:02d}:{m:02d} IST): {cnt:>6} records")


# ============================================================
# MAIN
# ============================================================
if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for station_id, file_dict in FILES.items():
        # Step 1: Merge files
        merged = merge_station_files(station_id, file_dict)

        # Step 2: Convert datetime
        merged = convert_datetime(merged)

        # Step 3: Reorder columns — put DATETIME_IST and DATETIME_UTC first
        front_cols = ["INDEX", "YEAR", "MN", "DT", "HR", "DATETIME_IST", "DATETIME_UTC"]
        other_cols = [c for c in merged.columns if c not in front_cols]
        merged = merged[front_cols + other_cols]

        # Step 4: Print summary
        print_summary(merged, station_id)

        # Step 5: Save
        out_path = os.path.join(OUTPUT_DIR, f"{station_id}_Table3_merged_1969_2025.csv")
        merged.to_csv(out_path, index=False)
        print(f"\n  ✅ Saved: {out_path}")
        print(f"     Total rows: {len(merged)}")
        print(f"     Columns: {list(merged.columns)}")

    print(f"\n{'='*60}")
    print(f"  DONE! Both stations merged successfully.")
    print(f"{'='*60}")



  Processing Station: 43003

  Loading 1969-2009 file...


/tmp/ipykernel_2128/2849744502.py:69: DtypeWarning: Columns (17,19,21,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


    Rows: 118157, Years: 1969-2009
  Loading 2010-2024 file...
    Rows: 35276, Years: 2010-2024
  Loading 2020-2025 file...
    Rows: 10408, Years: 2020-2025

  Part 1 (1969-2009): 118157 rows
  Part 2 (2010-2019): 26825 rows
  Part 3 (2020-2025) before dedup: 18859 rows
  Part 3 (2020-2025) after dedup:  10408 rows

  Total merged rows: 155390

  Converting HR codes to IST datetime...
    Date range: 1969-01-01 05:30:00 to 2025-06-01 02:30:00
    Valid datetimes: 155390 / 155390

  ──────────────────────────────────────────────────
  SUMMARY FOR STATION 43003
  ──────────────────────────────────────────────────

  Records per year:
    1969:  2916 records
    1970:  2915 records
    1971:  2917 records
    1972:  2924 records
    1973:  2917 records
    1974:  2914 records
    1975:  2918 records
    1976:  2914 records
    1977:  2911 records
    1978:  2916 records
    1979:  2919 records
    1980:  2915 records
    1981:  2906 records
    1982:  2909 records
    1983:  2917 record

/tmp/ipykernel_2128/2849744502.py:69: DtypeWarning: Columns (17,19,21,25) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


    Rows: 44753, Years: 1969-2009
  Loading 2010-2024 file...
    Rows: 13381, Years: 2010-2024
  Loading 2020-2025 file...
    Rows: 5391, Years: 2020-2025

  Part 1 (1969-2009): 44753 rows
  Part 2 (2010-2019): 9765 rows
  Part 3 (2020-2025) before dedup: 9007 rows
  Part 3 (2020-2025) after dedup:  5391 rows

  Total merged rows: 59909

  Converting HR codes to IST datetime...
    Date range: 1969-01-01 08:30:00 to 2025-05-31 20:30:00
    Valid datetimes: 59909 / 59909

  ──────────────────────────────────────────────────
  SUMMARY FOR STATION 43057
  ──────────────────────────────────────────────────

  Records per year:
    1969:  1154 records
    1970:  1089 records
    1971:  1093 records
    1972:  1096 records
    1973:  1092 records
    1974:  1095 records
    1975:  1093 records
    1976:  1094 records
    1977:  1089 records
    1978:  1092 records
    1979:  1094 records
    1980:  1097 records
    1981:  1093 records
    1982:  1089 records
    1983:  1095 records
    198

In [ ]:
"""
===============================================================================
  IMPUTATION PIPELINE — Colaba (43057)
  10 Methods | Outlier Treatment | Evaluation (MSE, RMSE, MAE, R²)

  Best Result: Weighted KNN-Mean-Interpolation WITH outlier treatment (R²=0.9625)
===============================================================================
"""

import pandas as pd
import numpy as np
import warnings
import os
import time
import gc

from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
INPUT_DIR = "/content/drive/MyDrive/Major_project_imd"       # Change to your path
OUTPUT_DIR = "/content/drive/MyDrive/Major_project_imd/imputation_outputs"       # Change to your path
os.makedirs(OUTPUT_DIR, exist_ok=True)

INPUT_FILE = os.path.join(INPUT_DIR, "43057_Table3_merged_1969_2025.csv")
STATION_ID = "43057"

TARGET_COLS = ["DBT", "WBT", "DPT", "RH", "VP", "FFF"]
EVAL_SAMPLE = 10000   # Rows to sample for evaluation
MASK_FRACTION = 0.05  # 5% masked for testing
SEED = 42
KNN_K = 5


# ============================================================
# UTILITY FUNCTIONS
# ============================================================

def load_data(filepath):
    """Load CSV and convert columns to numeric. Derive VP from DPT."""
    df = pd.read_csv(filepath, low_memory=False)
    for col in TARGET_COLS + ["MN", "HR", "SLP", "MSLP"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    # Derive VP from DPT using Magnus formula where VP is missing
    mask = df['VP'].isna() & df['DPT'].notna()
    df.loc[mask, 'VP'] = 6.112 * np.exp(
        (17.67 * df.loc[mask, 'DPT']) / (df.loc[mask, 'DPT'] + 243.5)
    )
    return df


def treat_outliers_iqr(df, columns, factor=1.5):
    """Replace outliers (IQR method) with NaN."""
    df_out = df.copy()
    counts = {}
    for col in columns:
        s = df_out[col].dropna()
        Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
        IQR = Q3 - Q1
        mask = (df_out[col] < Q1 - factor * IQR) | (df_out[col] > Q3 + factor * IQR)
        counts[col] = mask.sum()
        df_out.loc[mask, col] = np.nan
    return df_out, counts


def create_mask(df, columns, fraction=0.05, seed=42):
    """Mask known values for evaluation. Returns masked_df and truth dict."""
    rng = np.random.RandomState(seed)
    masked = df.copy()
    truth = {}
    for col in columns:
        idx = df[col].dropna().index.tolist()
        n = max(1, int(len(idx) * fraction))
        sel = rng.choice(idx, size=n, replace=False)
        truth[col] = df.loc[sel, col].to_dict()
        masked.loc[sel, col] = np.nan
    total = sum(len(v) for v in truth.values())
    print(f"\n  Masked {total} values for evaluation ({fraction*100:.0f}%)")
    return masked, truth


def get_feature_matrix(df):
    """Extract feature columns for imputation."""
    cols = TARGET_COLS.copy()
    for c in ["MN", "HR"]:
        if c in df.columns:
            cols.append(c)
    return df[cols].copy(), cols


# ============================================================
# BASE IMPUTATION METHODS
# ============================================================

def impute_mean(data, tcols):
    r = data.copy()
    for c in tcols:
        r[c] = r[c].fillna(r[c].mean())
    return r


def impute_median(data, tcols):
    r = data.copy()
    for c in tcols:
        r[c] = r[c].fillna(r[c].median())
    return r


def impute_interpolation(data, tcols):
    r = data.copy()
    for c in tcols:
        r[c] = r[c].interpolate(method='linear', limit_direction='both')
    return r


def impute_knn(data, all_cols, tcols):
    imp = KNNImputer(n_neighbors=KNN_K)
    arr = imp.fit_transform(data[all_cols].values)
    r = data.copy()
    for i, c in enumerate(all_cols):
        if c in tcols:
            r[c] = arr[:, i]
    return r


def impute_mice(data, all_cols, tcols):
    imp = IterativeImputer(max_iter=10, random_state=SEED, sample_posterior=False)
    arr = imp.fit_transform(data[all_cols].values)
    r = data.copy()
    for i, c in enumerate(all_cols):
        if c in tcols:
            r[c] = arr[:, i]
    return r


# ============================================================
# COMPUTE ALL BASE METHODS (cached for reuse by hybrids)
# ============================================================

def compute_base_methods(data, feat_cols):
    """Compute 5 base imputation results. Returns dict of DataFrames."""
    cache = {}

    t0 = time.time()
    cache["mean"] = impute_mean(data, TARGET_COLS)
    print(f"      Mean: {time.time()-t0:.1f}s")

    t0 = time.time()
    cache["median"] = impute_median(data, TARGET_COLS)
    print(f"      Median: {time.time()-t0:.1f}s")

    t0 = time.time()
    cache["interp"] = impute_interpolation(data, TARGET_COLS)
    print(f"      Interpolation: {time.time()-t0:.1f}s")

    t0 = time.time()
    cache["knn"] = impute_knn(data, feat_cols, TARGET_COLS)
    print(f"      KNN: {time.time()-t0:.1f}s")

    t0 = time.time()
    cache["mice"] = impute_mice(data, feat_cols, TARGET_COLS)
    print(f"      MICE: {time.time()-t0:.1f}s")

    return cache


# ============================================================
# 10 METHODS FROM CACHED BASE RESULTS
# ============================================================

def build_all_methods(data, cache):
    """Build all 10 imputed DataFrames from cached base methods."""
    methods = {}

    # 1-5: Direct base methods
    methods["1_Mean"] = cache["mean"]
    methods["2_Median"] = cache["median"]
    methods["3_Interpolation"] = cache["interp"]
    methods["4_KNN"] = cache["knn"]
    methods["5_MICE"] = cache["mice"]

    # 6: KNN-MICE Weighted (50/50)
    r = data.copy()
    for c in TARGET_COLS:
        r[c] = 0.5 * cache["knn"][c] + 0.5 * cache["mice"][c]
    methods["6_KNN_MICE"] = r

    # 7: Average KNN-Mean-Interpolation
    r = data.copy()
    for c in TARGET_COLS:
        r[c] = (cache["knn"][c] + cache["mean"][c] + cache["interp"][c]) / 3.0
    methods["7_Avg_KNN_Mean_Interp"] = r

    # 8: Weighted KNN-Mean-Interpolation (0.5/0.2/0.3)
    r = data.copy()
    for c in TARGET_COLS:
        r[c] = 0.5 * cache["knn"][c] + 0.2 * cache["mean"][c] + 0.3 * cache["interp"][c]
    methods["8_Weighted_KNN_Mean_Interp"] = r

    # 9: Dynamic Weighted (adapts based on local data availability)
    r = data.copy()
    for c in TARGET_COLS:
        avail = data[c].notna().astype(float)
        la = avail.rolling(11, center=True, min_periods=1).mean()
        wi = np.where(la > 0.6, 0.50, np.where(la > 0.3, 0.25, 0.10))
        wk = np.where(la > 0.6, 0.30, np.where(la > 0.3, 0.35, 0.35))
        wm = np.where(la > 0.6, 0.15, np.where(la > 0.3, 0.30, 0.40))
        wn = np.where(la > 0.6, 0.05, np.where(la > 0.3, 0.10, 0.15))
        blended = (wn * cache["mean"][c].values +
                   wi * cache["interp"][c].values +
                   wk * cache["knn"][c].values +
                   wm * cache["mice"][c].values)
        miss = data[c].isna()
        r[c] = data[c].copy()
        r.loc[miss, c] = blended[miss]
    methods["9_Dynamic_Weighted"] = r

    # 10: Column-wise Weighted (per-column optimal weights via validation)
    r = data.copy()
    rng2 = np.random.RandomState(SEED + 999)
    for c in TARGET_COLS:
        known = data[c].dropna().index.tolist()
        if len(known) < 100:
            r[c] = 0.25 * (cache["mean"][c] + cache["interp"][c] +
                           cache["knn"][c] + cache["mice"][c])
            continue
        vi = rng2.choice(known, size=min(500, len(known) // 10), replace=False)
        tv = data.loc[vi, c].values
        errs = {}
        for mn, bd in [("mean", cache["mean"]), ("interp", cache["interp"]),
                       ("knn", cache["knn"]), ("mice", cache["mice"])]:
            rmse = np.sqrt(mean_squared_error(tv, bd.loc[vi, c].values))
            errs[mn] = 1.0 / max(rmse, 1e-6)
        tot = sum(errs.values())
        w = {m: v / tot for m, v in errs.items()}
        r[c] = (w["mean"] * cache["mean"][c] + w["interp"] * cache["interp"][c] +
                w["knn"] * cache["knn"][c] + w["mice"] * cache["mice"][c])
    methods["10_Columnwise_Weighted"] = r

    return methods


# ============================================================
# EVALUATION
# ============================================================

def evaluate_all_methods(methods_dict, truth, tag=""):
    """Evaluate all methods against masked ground truth."""
    results = []
    percol_results = []

    for name in sorted(methods_dict.keys()):
        imp = methods_dict[name]
        all_t, all_p = [], []

        for c in TARGET_COLS:
            if c not in truth or not truth[c]:
                continue
            idx = list(truth[c].keys())
            t = np.array([truth[c][i] for i in idx])
            p = imp.loc[idx, c].values.astype(float)
            valid = ~(np.isnan(t) | np.isnan(p))
            if valid.sum() == 0:
                continue
            tv, pv = t[valid], p[valid]

            percol_results.append({
                "Method": name, "Column": c,
                "MSE": round(mean_squared_error(tv, pv), 4),
                "RMSE": round(np.sqrt(mean_squared_error(tv, pv)), 4),
                "MAE": round(mean_absolute_error(tv, pv), 4),
                "R2": round(r2_score(tv, pv), 4) if len(tv) > 1 else 0
            })
            all_t.extend(tv)
            all_p.extend(pv)

        if all_t:
            at, ap = np.array(all_t), np.array(all_p)
            row = {
                "Method": name,
                "MSE": round(mean_squared_error(at, ap), 4),
                "RMSE": round(np.sqrt(mean_squared_error(at, ap)), 4),
                "MAE": round(mean_absolute_error(at, ap), 4),
                "R2_Score": round(r2_score(at, ap), 4)
            }
            results.append(row)
            print(f"    {name:>35s}: RMSE={row['RMSE']:.4f}  "
                  f"MAE={row['MAE']:.4f}  R²={row['R2_Score']:.4f}")

    comp = pd.DataFrame(results).sort_values("RMSE")
    percol = pd.DataFrame(percol_results)

    comp.to_csv(os.path.join(OUTPUT_DIR,
        f"{STATION_ID}_imputation_{tag}.csv"), index=False)
    percol.to_csv(os.path.join(OUTPUT_DIR,
        f"{STATION_ID}_imputation_percol_{tag}.csv"), index=False)

    return comp


# ============================================================
# APPLY BEST METHOD TO FULL DATA
# ============================================================

def apply_best_method(df, best_name, use_ot=False):
    """Apply the selected best imputation method to the full dataset."""

    if use_ot:
        work = df.copy()
        for col in TARGET_COLS:
            s = work[col].dropna()
            Q1, Q3 = s.quantile(0.25), s.quantile(0.75)
            IQR = Q3 - Q1
            work.loc[(work[col] < Q1 - 1.5 * IQR) |
                     (work[col] > Q3 + 1.5 * IQR), col] = np.nan
    else:
        work = df

    feat_data, feat_cols = get_feature_matrix(work)

    print("    Computing base methods on full data...")
    cache = compute_base_methods(feat_data, feat_cols)

    result = work.copy()

    if "Weighted_KNN_Mean_Interp" in best_name and "Avg" not in best_name:
        for c in TARGET_COLS:
            result[c] = (0.5 * cache["knn"][c] +
                         0.2 * cache["mean"][c] +
                         0.3 * cache["interp"][c])
    elif "Avg_KNN_Mean_Interp" in best_name:
        for c in TARGET_COLS:
            result[c] = (cache["knn"][c] + cache["mean"][c] +
                         cache["interp"][c]) / 3.0
    elif "Dynamic_Weighted" in best_name:
        for c in TARGET_COLS:
            avail = feat_data[c].notna().astype(float)
            la = avail.rolling(11, center=True, min_periods=1).mean()
            wi = np.where(la > 0.6, 0.50, np.where(la > 0.3, 0.25, 0.10))
            wk = np.where(la > 0.6, 0.30, np.where(la > 0.3, 0.35, 0.35))
            wm = np.where(la > 0.6, 0.15, np.where(la > 0.3, 0.30, 0.40))
            wn = np.where(la > 0.6, 0.05, np.where(la > 0.3, 0.10, 0.15))
            blended = (wn * cache["mean"][c].values +
                       wi * cache["interp"][c].values +
                       wk * cache["knn"][c].values +
                       wm * cache["mice"][c].values)
            miss = feat_data[c].isna()
            result[c] = work[c].copy()
            result.loc[miss, c] = blended[miss]
    elif "KNN_MICE" in best_name:
        for c in TARGET_COLS:
            result[c] = 0.5 * cache["knn"][c] + 0.5 * cache["mice"][c]
    elif "Columnwise_Weighted" in best_name:
        rng2 = np.random.RandomState(SEED + 999)
        for c in TARGET_COLS:
            known = feat_data[c].dropna().index.tolist()
            vi = rng2.choice(known, size=min(500, len(known) // 10), replace=False)
            tv = feat_data.loc[vi, c].values
            errs = {}
            for mn, bd in [("mean", cache["mean"]), ("interp", cache["interp"]),
                           ("knn", cache["knn"]), ("mice", cache["mice"])]:
                errs[mn] = 1.0 / max(np.sqrt(
                    mean_squared_error(tv, bd.loc[vi, c].values)), 1e-6)
            tot = sum(errs.values())
            w = {m: v / tot for m, v in errs.items()}
            result[c] = (w["mean"] * cache["mean"][c] +
                         w["interp"] * cache["interp"][c] +
                         w["knn"] * cache["knn"][c] +
                         w["mice"] * cache["mice"][c])
    elif "KNN" in best_name:
        for c in TARGET_COLS:
            result[c] = cache["knn"][c]
    elif "MICE" in best_name:
        for c in TARGET_COLS:
            result[c] = cache["mice"][c]
    elif "Interpolation" in best_name:
        for c in TARGET_COLS:
            result[c] = cache["interp"][c]
    elif "Median" in best_name:
        for c in TARGET_COLS:
            result[c] = cache["median"][c]
    else:  # Mean or fallback
        for c in TARGET_COLS:
            result[c] = cache["mean"][c]

    del cache
    gc.collect()
    return result


# ============================================================
# MAIN PIPELINE
# ============================================================

if __name__ == "__main__":
    print("=" * 70)
    print(f"  IMPUTATION PIPELINE — Station {STATION_ID} (Colaba)")
    print("=" * 70)

    # --- Load data ---
    df = load_data(INPUT_FILE)
    print(f"\n  Loaded: {len(df)} rows")
    for c in TARGET_COLS:
        m = df[c].isna().sum()
        print(f"    {c}: {m} missing ({m / len(df) * 100:.2f}%)")

    # ===== PASS 1: WITHOUT OUTLIER TREATMENT =====
    print(f"\n{'='*60}")
    print(f"  PASS 1: WITHOUT OUTLIER TREATMENT")
    print(f"{'='*60}")

    masked, truth = create_mask(df, TARGET_COLS, MASK_FRACTION, SEED)
    feat_data, feat_cols = get_feature_matrix(masked)

    print("\n    Computing base methods...")
    cache = compute_base_methods(feat_data, feat_cols)
    methods_dict = build_all_methods(feat_data, cache)

    print("\n    Evaluating all methods:")
    comp_no_ot = evaluate_all_methods(methods_dict, truth, tag="NO_OT")

    print(f"\n  Results WITHOUT outlier treatment (sorted by RMSE):")
    print(f"  {comp_no_ot.to_string(index=False)}")

    del cache, methods_dict, feat_data, masked
    gc.collect()

    # ===== PASS 2: WITH OUTLIER TREATMENT =====
    print(f"\n{'='*60}")
    print(f"  PASS 2: WITH OUTLIER TREATMENT (IQR×1.5)")
    print(f"{'='*60}")

    df_ot, outlier_counts = treat_outliers_iqr(df, TARGET_COLS)
    print(f"\n  Outliers replaced with NaN:")
    for col, cnt in outlier_counts.items():
        print(f"    {col}: {cnt} outliers ({cnt / len(df) * 100:.2f}%)")

    masked_ot, truth_ot = create_mask(df_ot, TARGET_COLS, MASK_FRACTION, SEED)
    feat_data_ot, feat_cols_ot = get_feature_matrix(masked_ot)

    print("\n    Computing base methods...")
    cache_ot = compute_base_methods(feat_data_ot, feat_cols_ot)
    methods_dict_ot = build_all_methods(feat_data_ot, cache_ot)

    print("\n    Evaluating all methods:")
    comp_ot = evaluate_all_methods(methods_dict_ot, truth_ot, tag="WITH_OT")

    print(f"\n  Results WITH outlier treatment (sorted by RMSE):")
    print(f"  {comp_ot.to_string(index=False)}")

    del cache_ot, methods_dict_ot, feat_data_ot, masked_ot
    gc.collect()

    # ===== SELECT BEST METHOD =====
    print(f"\n{'='*60}")
    print(f"  BEST METHOD SELECTION")
    print(f"{'='*60}")

    best_no = comp_no_ot.iloc[0]
    best_ot = comp_ot.iloc[0]

    print(f"    Without outlier: {best_no['Method']} "
          f"(R²={best_no['R2_Score']}, RMSE={best_no['RMSE']})")
    print(f"    With outlier:    {best_ot['Method']} "
          f"(R²={best_ot['R2_Score']}, RMSE={best_ot['RMSE']})")

    if best_ot['R2_Score'] >= best_no['R2_Score']:
        best_name = best_ot['Method']
        use_ot = True
    else:
        best_name = best_no['Method']
        use_ot = False

    print(f"\n  ➤ SELECTED: {best_name} "
          f"{'WITH' if use_ot else 'WITHOUT'} outlier treatment")

    # ===== APPLY BEST TO FULL DATA =====
    print(f"\n{'='*60}")
    print(f"  APPLYING BEST METHOD TO FULL DATASET")
    print(f"{'='*60}")

    final = apply_best_method(df, best_name, use_ot)

    print(f"\n  Final missing values check:")
    for c in TARGET_COLS:
        rem = final[c].isna().sum()
        print(f"    {c}: {rem} remaining NaN")

    out_path = os.path.join(OUTPUT_DIR,
        f"{STATION_ID}_Table3_imputed_1969_2025.csv")
    final.to_csv(out_path, index=False)
    print(f"\n  ✅ Saved: {out_path} ({len(final)} rows)")

    print(f"\n{'='*60}")
    print(f"  DONE!")
    print(f"{'='*60}")


  IMPUTATION PIPELINE — Station 43057 (Colaba)

  Loaded: 59909 rows
    DBT: 9 missing (0.02%)
    WBT: 1893 missing (3.16%)
    DPT: 57 missing (0.10%)
    RH: 11 missing (0.02%)
    VP: 22 missing (0.04%)
    FFF: 132 missing (0.22%)

  PASS 1: WITHOUT OUTLIER TREATMENT

  Masked 17863 values for evaluation (5%)

    Computing base methods...
      Mean: 0.0s
      Median: 0.0s
      Interpolation: 0.0s
      KNN: 73.2s
      MICE: 1.8s

    Evaluating all methods:
                 10_Columnwise_Weighted: RMSE=3.4093  MAE=2.1433  R²=0.9756
                                 1_Mean: RMSE=7.1374  MAE=4.7913  R²=0.8929
                               2_Median: RMSE=7.2051  MAE=4.6784  R²=0.8909
                        3_Interpolation: RMSE=6.0691  MAE=3.4329  R²=0.9226
                                  4_KNN: RMSE=2.2408  MAE=1.0231  R²=0.9894
                                 5_MICE: RMSE=2.2844  MAE=1.0072  R²=0.9890
                             6_KNN_MICE: RMSE=2.0815  MAE=0.9090  R²=0.

In [ ]:
"""
Ultra-lean imputation for 43003 — minimal memory
"""
import pandas as pd
import numpy as np
import warnings, os, gc, time
from sklearn.impute import KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
warnings.filterwarnings('ignore')

OUTPUT_DIR = "/content/drive/MyDrive/Major_project_imd/imputation_outputs"
TCOLS = ["DBT","WBT","DPT","RH","VP","FFF"]
FCOLS = TCOLS + ["MN","HR"]
SEED = 42

# ========== STEP 1: EVALUATION ON SMALL SAMPLE ==========
print("=== STEP 1: Load only needed columns ===")
df_full = pd.read_csv("/content/drive/MyDrive/Major_project_imd/43003_Table3_merged_1969_2025.csv",
                       usecols=["YEAR","MN","HR","DT"] + TCOLS, low_memory=False)
for c in FCOLS + ["YEAR","DT"]:
    df_full[c] = pd.to_numeric(df_full[c], errors='coerce')

# Derive VP
m = df_full['VP'].isna() & df_full['DPT'].notna()
df_full.loc[m,'VP'] = 6.112*np.exp((17.67*df_full.loc[m,'DPT'])/(df_full.loc[m,'DPT']+243.5))

print(f"Rows: {len(df_full)}")
for c in TCOLS:
    n=df_full[c].isna().sum()
    print(f"  {c}: {n} missing ({n/len(df_full)*100:.2f}%)")

# Take 5k sample for evaluation
print("\n=== STEP 2: Evaluation on 5k sample ===")
complete = df_full[TCOLS].notna().all(axis=1)
rng = np.random.RandomState(SEED)
cidx = df_full.index[complete].tolist()
sidx = sorted(rng.choice(cidx, size=min(2000,len(cidx)), replace=False))

# Extract small evaluation dataset with context window
ctx = set()
for i in sidx:
    for j in range(max(0,i-10), min(len(df_full),i+11)):
        ctx.add(j)
ctx_idx = sorted(ctx)
sub = df_full.loc[ctx_idx, FCOLS].copy().reset_index(drop=True)

# Map indices
o2n = {old:new for new,old in enumerate(ctx_idx)}
mapped = [o2n[i] for i in sidx]

# Mask 5%
nm = int(len(mapped)*0.05)
midx = rng.choice(mapped, size=nm, replace=False)
truth = {}
esub = sub.copy()
for c in TCOLS:
    truth[c] = sub.loc[midx,c].to_dict()
    esub.loc[midx,c] = np.nan

print(f"  Context: {len(sub)} rows, Masked: {nm} per col")

# Compute 4 base methods
def run_bases(data):
    b = {}
    b["mean"] = data.copy()
    for c in TCOLS: b["mean"][c]=b["mean"][c].fillna(b["mean"][c].mean())

    b["interp"] = data.copy()
    for c in TCOLS: b["interp"][c]=b["interp"][c].interpolate(method='linear',limit_direction='both')

    t0=time.time()
    arr = KNNImputer(n_neighbors=5).fit_transform(data.values)
    b["knn"] = pd.DataFrame(arr, columns=data.columns, index=data.index)
    print(f"    KNN: {time.time()-t0:.1f}s")

    t0=time.time()
    arr = IterativeImputer(max_iter=10,random_state=SEED).fit_transform(data.values)
    b["mice"] = pd.DataFrame(arr, columns=data.columns, index=data.index)
    print(f"    MICE: {time.time()-t0:.1f}s")
    return b

def eval_all(bases, data, truth_dict, tag):
    methods = {
        "1_Mean": lambda: bases["mean"],
        "2_Median": lambda: (lambda d: (([d.__setitem__(c, d[c].fillna(d[c].median())) for c in TCOLS]), d)[1])(data.copy()),
        "3_Interpolation": lambda: bases["interp"],
        "4_KNN": lambda: bases["knn"],
        "5_MICE": lambda: bases["mice"],
    }
    # Hybrids
    def m6():
        r=data.copy()
        for c in TCOLS: r[c]=0.5*bases["knn"][c]+0.5*bases["mice"][c]
        return r
    def m7():
        r=data.copy()
        for c in TCOLS: r[c]=(bases["knn"][c]+bases["mean"][c]+bases["interp"][c])/3
        return r
    def m8():
        r=data.copy()
        for c in TCOLS: r[c]=0.5*bases["knn"][c]+0.2*bases["mean"][c]+0.3*bases["interp"][c]
        return r
    def m9():
        r=data.copy()
        for c in TCOLS:
            la=data[c].notna().astype(float).rolling(11,center=True,min_periods=1).mean()
            wi=np.where(la>0.6,0.50,np.where(la>0.3,0.25,0.10))
            wk=np.where(la>0.6,0.30,np.where(la>0.3,0.35,0.35))
            wm=np.where(la>0.6,0.15,np.where(la>0.3,0.30,0.40))
            wn=np.where(la>0.6,0.05,np.where(la>0.3,0.10,0.15))
            bl=wn*bases["mean"][c].values+wi*bases["interp"][c].values+wk*bases["knn"][c].values+wm*bases["mice"][c].values
            miss=data[c].isna(); r[c]=data[c].copy(); r.loc[miss,c]=bl[miss]
        return r
    def m10():
        r=data.copy(); rng2=np.random.RandomState(SEED+999)
        for c in TCOLS:
            kn=data[c].dropna().index.tolist()
            vi=rng2.choice(kn,size=min(300,len(kn)//10),replace=False)
            tv=data.loc[vi,c].values; errs={}
            for mn,bd in [("mean",bases["mean"]),("interp",bases["interp"]),("knn",bases["knn"]),("mice",bases["mice"])]:
                errs[mn]=1.0/max(np.sqrt(mean_squared_error(tv,bd.loc[vi,c].values)),1e-6)
            tot=sum(errs.values()); w={m:v/tot for m,v in errs.items()}
            r[c]=w["mean"]*bases["mean"][c]+w["interp"]*bases["interp"][c]+w["knn"]*bases["knn"][c]+w["mice"]*bases["mice"][c]
        return r

    methods.update({"6_KNN_MICE":m6,"7_Avg_KNN_Mean_Interp":m7,
                    "8_Weighted_KNN_Mean_Interp":m8,"9_Dynamic_Weighted":m9,"10_Columnwise_Weighted":m10})

    rows = []
    for name in sorted(methods.keys()):
        imp = methods[name]()
        at,ap=[],[]
        for c in TCOLS:
            idx=list(truth_dict[c].keys())
            t=np.array([truth_dict[c][i] for i in idx])
            p=imp.loc[idx,c].values.astype(float)
            v=~(np.isnan(t)|np.isnan(p))
            if v.sum()>0: at.extend(t[v]); ap.extend(p[v])
        at,ap=np.array(at),np.array(ap)
        rows.append({"Method":name,"MSE":round(mean_squared_error(at,ap),4),
                     "RMSE":round(np.sqrt(mean_squared_error(at,ap)),4),
                     "MAE":round(mean_absolute_error(at,ap),4),
                     "R2_Score":round(r2_score(at,ap),4)})
        del imp

    result = pd.DataFrame(rows).sort_values("RMSE")
    print(f"\n  {tag}:")
    for _,r in result.iterrows():
        print(f"    {r['Method']:>35s}: RMSE={r['RMSE']:.4f}  MAE={r['MAE']:.4f}  R²={r['R2_Score']:.4f}")
    return result

# Pass 1: No outlier
print("  Computing bases (no OT)...")
bases1 = run_bases(esub)
comp1 = eval_all(bases1, esub, truth, "WITHOUT outlier treatment")
comp1.to_csv(os.path.join(OUTPUT_DIR,"43003_imputation_NO_OT.csv"),index=False)
del bases1; gc.collect()

# Pass 2: With outlier
print("\n  Outlier treatment...")
esub_ot = esub.copy()
for c in TCOLS:
    s=esub_ot[c].dropna(); Q1,Q3=s.quantile(0.25),s.quantile(0.75); IQR=Q3-Q1
    om=(esub_ot[c]<Q1-1.5*IQR)|(esub_ot[c]>Q3+1.5*IQR)
    print(f"    {c}: {om.sum()} outliers")
    esub_ot.loc[om,c]=np.nan

truth_ot = {}
for c in TCOLS:
    truth_ot[c] = {k:v for k,v in truth[c].items() if not np.isnan(esub_ot.loc[k,c]) if k in esub_ot.index}
    # re-mask
    for k in truth[c]: esub_ot.loc[k,c] = np.nan
truth_ot = truth  # use original truth for fair comparison

print("  Computing bases (with OT)...")
bases2 = run_bases(esub_ot)
comp2 = eval_all(bases2, esub_ot, truth_ot, "WITH outlier treatment")
comp2.to_csv(os.path.join(OUTPUT_DIR,"43003_imputation_WITH_OT.csv"),index=False)
del bases2, esub, esub_ot, sub; gc.collect()

# ========== STEP 3: APPLY BEST TO FULL DATA ==========
b1,b2 = comp1.iloc[0], comp2.iloc[0]
if b2['R2_Score']>=b1['R2_Score']:
    best,use_ot = b2['Method'],True
else:
    best,use_ot = b1['Method'],False

print(f"\n➤ BEST: {best} {'WITH' if use_ot else 'WITHOUT'} OT (R²={max(b1['R2_Score'],b2['R2_Score'])})")

# Apply outlier treatment if needed
if use_ot:
    for c in TCOLS:
        s=df_full[c].dropna(); Q1,Q3=s.quantile(0.25),s.quantile(0.75); IQR=Q3-Q1
        df_full.loc[(df_full[c]<Q1-1.5*IQR)|(df_full[c]>Q3+1.5*IQR),c]=np.nan

feat = df_full[FCOLS].copy()

# Mean & Interpolation (fast)
print("\nFull data: Mean...")
f_mean = feat.copy()
for c in TCOLS: f_mean[c]=f_mean[c].fillna(f_mean[c].mean())

print("Full data: Interpolation...")
f_interp = feat.copy()
for c in TCOLS: f_interp[c]=f_interp[c].interpolate(method='linear',limit_direction='both')

# Chunked KNN
print("Full data: Chunked KNN...")
f_knn_vals = {c: np.full(len(feat), np.nan) for c in TCOLS}
chunk = 25000; overlap = 1000

for start in range(0, len(feat), chunk):
    cs = max(0,start-overlap)
    ce = min(len(feat),start+chunk+overlap)
    ch = feat.iloc[cs:ce].copy()
    arr = KNNImputer(n_neighbors=5).fit_transform(ch.values)
    res = pd.DataFrame(arr, columns=FCOLS)

    ws = start-cs  # write start within chunk
    we = ws + min(chunk, len(feat)-start)
    for ic,c in enumerate(FCOLS):
        if c in TCOLS:
            f_knn_vals[c][start:start+(we-ws)] = res.iloc[ws:we, ic].values

    del ch, arr, res; gc.collect()
    print(f"  Chunk {start}-{min(start+chunk,len(feat))}")

# Apply best formula
print(f"\nApplying formula: {best}")
for c in TCOLS:
    knn_col = f_knn_vals[c]
    if "Weighted_KNN_Mean_Interp" in best:
        df_full[c] = 0.5*knn_col + 0.2*f_mean[c].values + 0.3*f_interp[c].values
    elif "Dynamic_Weighted" in best:
        la=feat[c].notna().astype(float).rolling(11,center=True,min_periods=1).mean().values
        wi=np.where(la>0.6,0.50,np.where(la>0.3,0.25,0.10))
        wk=np.where(la>0.6,0.30,np.where(la>0.3,0.35,0.35))
        wn=np.where(la>0.6,0.05,np.where(la>0.3,0.10,0.15))
        # Need MICE too for dynamic - use mean as substitute to save memory
        wm=np.where(la>0.6,0.15,np.where(la>0.3,0.30,0.40))
        bl=wn*f_mean[c].values+wi*f_interp[c].values+wk*knn_col+wm*f_mean[c].values
        miss=feat[c].isna()
        vals = feat[c].copy().values
        vals[miss]=bl[miss]
        df_full[c]=vals
    elif "Avg_KNN_Mean_Interp" in best:
        df_full[c] = (knn_col+f_mean[c].values+f_interp[c].values)/3
    elif "KNN" in best:
        df_full[c] = knn_col
    else:
        df_full[c] = f_mean[c].values

del f_mean, f_interp, f_knn_vals, feat; gc.collect()

print("\nRemaining NaN:")
for c in TCOLS:
    r = df_full[c].isna().sum()
    print(f"  {c}: {r}")

# Reload full CSV and replace target columns
print("\nReloading full CSV to preserve all columns...")
final = pd.read_csv("/content/drive/MyDrive/Major_project_imd/43003_Table3_merged_1969_2025.csv", low_memory=False)
for c in TCOLS:
    final[c] = df_full[c].values

out = os.path.join(OUTPUT_DIR, "43003_Table3_imputed_1969_2025.csv")
final.to_csv(out, index=False)
print(f"\n✅ Saved: {out} ({len(final)} rows)")

=== STEP 1: Load only needed columns ===
Rows: 155390
  DBT: 12 missing (0.01%)
  WBT: 2966 missing (1.91%)
  DPT: 264 missing (0.17%)
  RH: 26 missing (0.02%)
  VP: 87 missing (0.06%)
  FFF: 49 missing (0.03%)

=== STEP 2: Evaluation on 5k sample ===
  Context: 36991 rows, Masked: 100 per col
  Computing bases (no OT)...
    KNN: 0.6s
    MICE: 1.1s

  WITHOUT outlier treatment:
                        3_Interpolation: RMSE=3.8021  MAE=2.2173  R²=0.9682
                     9_Dynamic_Weighted: RMSE=4.4997  MAE=2.8150  R²=0.9554
             8_Weighted_KNN_Mean_Interp: RMSE=4.9041  MAE=3.0891  R²=0.9471
                  7_Avg_KNN_Mean_Interp: RMSE=5.2398  MAE=3.3623  R²=0.9396
                                  4_KNN: RMSE=5.6473  MAE=3.4670  R²=0.9298
                 10_Columnwise_Weighted: RMSE=6.0693  MAE=3.9903  R²=0.9189
                             6_KNN_MICE: RMSE=6.6334  MAE=4.3261  R²=0.9031
                                 1_Mean: RMSE=8.9678  MAE=6.0952  R²=0.8230
         

In [ ]:
!pip install imbalanced-learn pythermalcomfort scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 229.9/229.9 kB 8.8 MB/s eta 0:00:00


In [ ]:
"""
===============================================================================
  HEAT INDEX & PET COMPUTATION + CLASS BALANCING PIPELINE

  Step 1: Compute Heat Index (HI) using Rothfusz/NOAA equation
  Step 2: Compute PET using pythermalcomfort (with Tmrt estimation)
  Step 3: Assign heat stress categories (classification labels)
  Step 4: Class balancing using 7 techniques (SMOTE + others)
  Step 5: Comparison of balancing techniques

  Extending Desai et al. (2021) — treating as classification problem
===============================================================================
"""

import pandas as pd
import numpy as np
import warnings
import os
import time
import gc
from collections import Counter

warnings.filterwarnings('ignore')

# ============================================================
# CONFIGURATION
# ============================================================
# UPDATE THESE PATHS for your environment (Colab/local)
INPUT_DIR = "/content/drive/MyDrive/Major_project_imd/imputation_outputs"     # Where imputed files are
OUTPUT_DIR = "/content/drive/MyDrive/Major_project_imd/Class_balancing_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)

FILES = {
    "43003": os.path.join(INPUT_DIR, "43003_Table3_imputed_1969_2025.csv"),
    "43057": os.path.join(INPUT_DIR, "43057_Table3_imputed_1969_2025.csv"),
}

# Standard person parameters for PET (from the paper)
PET_MET = 1.4    # Metabolic rate in met (≈80W light activity)
PET_CLO = 0.9    # Clothing insulation in clo


# ============================================================
# STEP 1: HEAT INDEX COMPUTATION (Rothfusz/NOAA equation)
# ============================================================

def compute_heat_index(T_celsius, RH):
    """
    Compute Heat Index using NOAA/Rothfusz equation with adjustments.

    Parameters:
        T_celsius: Dry bulb temperature in °C
        RH: Relative humidity in %
    Returns:
        Heat Index in °C

    Reference: Rothfusz (1990), NOAA-NWS adjustments
    """
    # Convert to Fahrenheit for the equation
    T = T_celsius * 9.0 / 5.0 + 32.0

    # Simple formula for low HI values
    HI_simple = 0.5 * (T + 61.0 + ((T - 68.0) * 1.2) + (RH * 0.094))

    # Full Rothfusz regression
    HI = (-42.379
          + 2.04901523 * T
          + 10.14333127 * RH
          - 0.22475541 * T * RH
          - 6.83783e-3 * T**2
          - 5.481717e-2 * RH**2
          + 1.22874e-3 * T**2 * RH
          + 8.5282e-4 * T * RH**2
          - 1.99e-6 * T**2 * RH**2)

    # Use simple formula when average of simple HI and T < 80°F
    use_simple = (HI_simple + T) / 2.0 < 80.0
    HI = np.where(use_simple, HI_simple, HI)

    # NOAA Adjustment 1: Low humidity, high temperature
    adj1_mask = (RH < 13) & (T >= 80) & (T <= 112)
    adj1 = -((13 - RH) / 4.0) * np.sqrt((17 - np.abs(T - 95)) / 17.0)
    HI = np.where(adj1_mask, HI + adj1, HI)

    # NOAA Adjustment 2: High humidity, moderate temperature
    adj2_mask = (RH > 85) & (T >= 80) & (T <= 87)
    adj2 = ((RH - 85) / 10.0) * ((87 - T) / 5.0)
    HI = np.where(adj2_mask, HI + adj2, HI)

    # Convert back to Celsius
    HI_celsius = (HI - 32.0) * 5.0 / 9.0

    return HI_celsius


# ============================================================
# STEP 2: PET COMPUTATION (with Tmrt estimation)
# ============================================================

def estimate_tmrt(tdb, tc, hr):
    """
    Estimate Mean Radiant Temperature (Tmrt) from air temperature,
    cloud cover (oktas), and hour code.

    Simplified outdoor estimation:
    - Daytime (HR 24,36,48 → 11:30-17:30 IST): Higher Tmrt due to solar radiation
    - Morning/Evening (HR 12,60 → 08:30, 20:30 IST): Moderate Tmrt
    - Night (HR 0,72,84 → 05:30, 23:30, 02:30 IST): Tmrt ≈ Ta

    Cloud cover reduces Tmrt: more clouds = less radiation = lower Tmrt
    """
    tc = np.clip(np.where(np.isnan(tc), 4, tc), 0, 8)  # default 4 oktas if missing
    cloud_factor = 1.0 - (tc / 8.0) * 0.7  # 0.3 to 1.0 (overcast to clear)

    # Time-of-day solar factor
    hr = np.array(hr, dtype=float)
    solar_factor = np.where(
        np.isin(hr, [24, 36, 48]),  # Daytime peak (11:30-17:30)
        20.0,
        np.where(
            np.isin(hr, [12, 60]),   # Morning/Evening
            10.0,
            2.0                       # Night
        )
    )

    tmrt = tdb + solar_factor * cloud_factor
    return tmrt


def compute_pet_batch(tdb, tmrt, v_kmh, rh, batch_size=5000):
    """
    Compute PET in batches using pythermalcomfort.

    Parameters:
        tdb: Air temperature (°C)
        tmrt: Mean radiant temperature (°C)
        v_kmh: Wind speed in km/h (converted to m/s internally)
        rh: Relative humidity (%)
    Returns:
        PET values in °C
    """
    from pythermalcomfort.models import pet_steady

    # Convert wind speed: km/h → m/s, minimum 0.1 m/s
    v_ms = np.maximum(v_kmh / 3.6, 0.1)

    n = len(tdb)
    pet_values = np.full(n, np.nan)

    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        for i in range(start, end):
            try:
                result = pet_steady(
                    tdb=float(tdb[i]),
                    tr=float(tmrt[i]),
                    v=float(v_ms[i]),
                    rh=float(rh[i]),
                    met=PET_MET,
                    clo=PET_CLO
                )
                pet_values[i] = result.pet
            except:
                pet_values[i] = np.nan

        pct = end / n * 100
        if pct % 20 < (batch_size / n * 100):
            print(f"      PET computation: {pct:.0f}% done...")

    return pet_values


# ============================================================
# STEP 3: CATEGORY ASSIGNMENT
# ============================================================

def assign_hi_category(hi_values):
    """
    Assign Heat Index categories (from the paper Table 1):
        Low risk:       HI < 33
        Moderate risk:  33 ≤ HI < 39
        High risk:      39 ≤ HI < 46
        Very high risk: HI ≥ 46
    """
    categories = np.where(
        hi_values < 33, 0,           # Low risk
        np.where(
            hi_values < 39, 1,       # Moderate risk
            np.where(
                hi_values < 46, 2,   # High risk
                3                     # Very high risk
            )
        )
    )
    return categories


def assign_pet_category(pet_values):
    """
    Assign PET categories (from the paper Table 2):
        Slightly warm:  23 ≤ PET < 29  (Slight heat stress)
        Warm:           29 ≤ PET < 35  (Moderate heat stress)
        Hot:            35 ≤ PET < 41  (Strong heat stress)
        Very hot:       PET ≥ 41       (Extreme heat stress)

    Note: PET < 23 is 'Comfortable' (no heat stress) — added as class 0
    """
    categories = np.where(
        pet_values < 23, 0,          # Comfortable (no heat stress)
        np.where(
            pet_values < 29, 1,      # Slightly warm
            np.where(
                pet_values < 35, 2,  # Warm
                np.where(
                    pet_values < 41, 3,  # Hot
                    4                     # Very hot
                )
            )
        )
    )
    return categories


HI_CATEGORY_NAMES = {0: "Low_Risk", 1: "Moderate_Risk", 2: "High_Risk", 3: "Very_High_Risk"}
PET_CATEGORY_NAMES = {0: "Comfortable", 1: "Slightly_Warm", 2: "Warm", 3: "Hot", 4: "Very_Hot"}


# ============================================================
# STEP 4: CLASS BALANCING TECHNIQUES
# ============================================================

def apply_balancing_techniques(X, y, target_name="HI"):
    """
    Apply 7 class balancing techniques and compare.

    Techniques:
        1. SMOTE (Synthetic Minority Over-sampling)
        2. ADASYN (Adaptive Synthetic Sampling)
        3. Borderline-SMOTE
        4. SMOTE-ENN (SMOTE + Edited Nearest Neighbors)
        5. SMOTE-Tomek (SMOTE + Tomek Links)
        6. Random Oversampling
        7. Random Undersampling

    Returns: dict of {method_name: (X_balanced, y_balanced, class_distribution)}
    """
    from imblearn.over_sampling import (
        SMOTE, ADASYN, BorderlineSMOTE, RandomOverSampler
    )
    from imblearn.combine import SMOTEENN, SMOTETomek
    from imblearn.under_sampling import RandomUnderSampler

    print(f"\n    Original {target_name} distribution:")
    orig_dist = Counter(y)
    for cls in sorted(orig_dist.keys()):
        print(f"      Class {cls}: {orig_dist[cls]} samples ({orig_dist[cls]/len(y)*100:.1f}%)")

    # Check minimum class size for k_neighbors
    min_class_size = min(orig_dist.values())
    k_neighbors = min(5, min_class_size - 1) if min_class_size > 1 else 1

    techniques = {}

    # 1. SMOTE
    try:
        print(f"\n    Applying SMOTE (k={k_neighbors})...")
        t0 = time.time()
        sm = SMOTE(random_state=42, k_neighbors=k_neighbors)
        X_res, y_res = sm.fit_resample(X, y)
        techniques["1_SMOTE"] = (X_res, y_res, Counter(y_res))
        print(f"      Done in {time.time()-t0:.1f}s — {len(y)} → {len(y_res)} samples")
    except Exception as e:
        print(f"      SMOTE failed: {e}")

    # 2. ADASYN
    try:
        print(f"    Applying ADASYN...")
        t0 = time.time()
        ada = ADASYN(random_state=42, n_neighbors=k_neighbors)
        X_res, y_res = ada.fit_resample(X, y)
        techniques["2_ADASYN"] = (X_res, y_res, Counter(y_res))
        print(f"      Done in {time.time()-t0:.1f}s — {len(y)} → {len(y_res)} samples")
    except Exception as e:
        print(f"      ADASYN failed: {e}")

    # 3. Borderline-SMOTE
    try:
        print(f"    Applying Borderline-SMOTE...")
        t0 = time.time()
        bsm = BorderlineSMOTE(random_state=42, k_neighbors=k_neighbors)
        X_res, y_res = bsm.fit_resample(X, y)
        techniques["3_Borderline_SMOTE"] = (X_res, y_res, Counter(y_res))
        print(f"      Done in {time.time()-t0:.1f}s — {len(y)} → {len(y_res)} samples")
    except Exception as e:
        print(f"      Borderline-SMOTE failed: {e}")

    # 4. SMOTE-ENN
    try:
        print(f"    Applying SMOTE-ENN...")
        t0 = time.time()
        se = SMOTEENN(random_state=42, smote=SMOTE(k_neighbors=k_neighbors, random_state=42))
        X_res, y_res = se.fit_resample(X, y)
        techniques["4_SMOTE_ENN"] = (X_res, y_res, Counter(y_res))
        print(f"      Done in {time.time()-t0:.1f}s — {len(y)} → {len(y_res)} samples")
    except Exception as e:
        print(f"      SMOTE-ENN failed: {e}")

    # 5. SMOTE-Tomek
    try:
        print(f"    Applying SMOTE-Tomek...")
        t0 = time.time()
        st = SMOTETomek(random_state=42, smote=SMOTE(k_neighbors=k_neighbors, random_state=42))
        X_res, y_res = st.fit_resample(X, y)
        techniques["5_SMOTE_Tomek"] = (X_res, y_res, Counter(y_res))
        print(f"      Done in {time.time()-t0:.1f}s — {len(y)} → {len(y_res)} samples")
    except Exception as e:
        print(f"      SMOTE-Tomek failed: {e}")

    # 6. Random Oversampling
    try:
        print(f"    Applying Random Oversampling...")
        t0 = time.time()
        ros = RandomOverSampler(random_state=42)
        X_res, y_res = ros.fit_resample(X, y)
        techniques["6_Random_Oversampling"] = (X_res, y_res, Counter(y_res))
        print(f"      Done in {time.time()-t0:.1f}s — {len(y)} → {len(y_res)} samples")
    except Exception as e:
        print(f"      Random Oversampling failed: {e}")

    # 7. Random Undersampling
    try:
        print(f"    Applying Random Undersampling...")
        t0 = time.time()
        rus = RandomUnderSampler(random_state=42)
        X_res, y_res = rus.fit_resample(X, y)
        techniques["7_Random_Undersampling"] = (X_res, y_res, Counter(y_res))
        print(f"      Done in {time.time()-t0:.1f}s — {len(y)} → {len(y_res)} samples")
    except Exception as e:
        print(f"      Random Undersampling failed: {e}")

    return techniques


def print_balancing_comparison(techniques, original_y, target_name="HI"):
    """Print comparison table of all balancing techniques."""
    orig = Counter(original_y)
    classes = sorted(orig.keys())

    print(f"\n  {'='*80}")
    print(f"  CLASS DISTRIBUTION COMPARISON — {target_name}")
    print(f"  {'='*80}")

    # Header
    header = f"  {'Method':<28s}"
    for cls in classes:
        header += f"  Class{cls}"
    header += f"  {'Total':>8s}  {'Ratio':>8s}"
    print(header)
    print(f"  {'-'*78}")

    # Original
    row = f"  {'Original':<28s}"
    for cls in classes:
        row += f"  {orig[cls]:>6d}"
    ratio = max(orig.values()) / max(min(orig.values()), 1)
    row += f"  {sum(orig.values()):>8d}  {ratio:>7.1f}x"
    print(row)

    # Each technique
    for name in sorted(techniques.keys()):
        _, _, dist = techniques[name]
        row = f"  {name:<28s}"
        for cls in classes:
            row += f"  {dist.get(cls, 0):>6d}"
        ratio = max(dist.values()) / max(min(dist.values()), 1) if dist else 0
        row += f"  {sum(dist.values()):>8d}  {ratio:>7.1f}x"
        print(row)


# ============================================================
# MAIN PIPELINE
# ============================================================

def process_station(station_id, filepath):
    print(f"\n{'#'*70}")
    print(f"#  Station {station_id}")
    print(f"{'#'*70}")

    # --- Load imputed data ---
    print(f"\n  Loading imputed data...")
    df = pd.read_csv(filepath, low_memory=False)
    for col in ["DBT", "WBT", "DPT", "RH", "VP", "FFF", "MN", "HR", "TC"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
    print(f"    Rows: {len(df)}")

    # --- Compute Heat Index ---
    print(f"\n  Computing Heat Index (HI)...")
    df["HI"] = compute_heat_index(df["DBT"].values, df["RH"].values)
    print(f"    HI range: {df['HI'].min():.1f} to {df['HI'].max():.1f} °C")
    print(f"    HI mean:  {df['HI'].mean():.1f} °C")

    # --- Compute PET ---
    print(f"\n  Computing PET...")
    print(f"    Estimating Tmrt from air temp, cloud cover, and time of day...")
    df["Tmrt"] = estimate_tmrt(df["DBT"].values, df["TC"].values, df["HR"].values)

    print(f"    Computing PET (this may take a few minutes)...")
    t0 = time.time()
    df["PET"] = compute_pet_batch(
        df["DBT"].values, df["Tmrt"].values,
        df["FFF"].values, df["RH"].values,
        batch_size=10000
    )
    print(f"    PET computation: {time.time()-t0:.1f}s")
    print(f"    PET range: {df['PET'].min():.1f} to {df['PET'].max():.1f} °C")
    print(f"    PET mean:  {df['PET'].mean():.1f} °C")
    print(f"    PET NaN:   {df['PET'].isna().sum()}")

    # Fill any remaining PET NaN with median
    if df["PET"].isna().sum() > 0:
        df["PET"] = df["PET"].fillna(df["PET"].median())

    # --- Assign Categories ---
    print(f"\n  Assigning heat stress categories...")
    df["HI_Category"] = assign_hi_category(df["HI"].values)
    df["PET_Category"] = assign_pet_category(df["PET"].values)

    print(f"\n    HI Category Distribution:")
    for cat, name in HI_CATEGORY_NAMES.items():
        count = (df["HI_Category"] == cat).sum()
        pct = count / len(df) * 100
        print(f"      {cat} ({name:>15s}): {count:>7d} ({pct:5.1f}%)")

    print(f"\n    PET Category Distribution:")
    for cat, name in PET_CATEGORY_NAMES.items():
        count = (df["PET_Category"] == cat).sum()
        pct = count / len(df) * 100
        print(f"      {cat} ({name:>15s}): {count:>7d} ({pct:5.1f}%)")

    # --- Save dataset with HI, PET, and categories ---
    hi_pet_path = os.path.join(OUTPUT_DIR, f"{station_id}_with_HI_PET_categories.csv")
    df.to_csv(hi_pet_path, index=False)
    print(f"\n  ✅ Saved with HI/PET: {hi_pet_path}")

    # --- Class Balancing ---
    # Feature columns for classification
    feature_cols = ["DBT", "WBT", "DPT", "RH", "VP", "FFF", "MN", "HR"]

    X = df[feature_cols].values
    y_hi = df["HI_Category"].values.astype(int)
    y_pet = df["PET_Category"].values.astype(int)

    # Remove any rows with NaN in features
    valid = ~np.isnan(X).any(axis=1)
    X = X[valid]
    y_hi = y_hi[valid]
    y_pet = y_pet[valid]
    print(f"\n  Valid samples for balancing: {len(X)}")

    # Balance HI categories
    print(f"\n{'='*70}")
    print(f"  CLASS BALANCING — HI Categories — Station {station_id}")
    print(f"{'='*70}")

    hi_techniques = apply_balancing_techniques(X, y_hi, "HI")
    print_balancing_comparison(hi_techniques, y_hi, "HI")

    # Balance PET categories
    print(f"\n{'='*70}")
    print(f"  CLASS BALANCING — PET Categories — Station {station_id}")
    print(f"{'='*70}")

    pet_techniques = apply_balancing_techniques(X, y_pet, "PET")
    print_balancing_comparison(pet_techniques, y_pet, "PET")

    # --- Save balanced datasets ---
    print(f"\n  Saving balanced datasets...")

    for target_name, techniques, y_orig in [("HI", hi_techniques, y_hi),
                                             ("PET", pet_techniques, y_pet)]:
        # Save comparison table
        comp_rows = []
        orig_dist = Counter(y_orig)
        comp_rows.append({"Method": "Original", "Total": len(y_orig),
                          **{f"Class_{c}": orig_dist[c] for c in sorted(orig_dist.keys())},
                          "Imbalance_Ratio": round(max(orig_dist.values())/max(min(orig_dist.values()),1), 2)})

        for name, (X_b, y_b, dist) in techniques.items():
            comp_rows.append({"Method": name, "Total": len(y_b),
                              **{f"Class_{c}": dist.get(c, 0) for c in sorted(orig_dist.keys())},
                              "Imbalance_Ratio": round(max(dist.values())/max(min(dist.values()),1), 2)})

        comp_df = pd.DataFrame(comp_rows)
        comp_path = os.path.join(OUTPUT_DIR,
            f"{station_id}_{target_name}_balancing_comparison.csv")
        comp_df.to_csv(comp_path, index=False)
        print(f"    ✅ {target_name} comparison: {comp_path}")

        # Save SMOTE-balanced dataset (most commonly used)
        if "1_SMOTE" in techniques:
            X_smote, y_smote, _ = techniques["1_SMOTE"]
            smote_df = pd.DataFrame(X_smote, columns=feature_cols)
            smote_df[f"{target_name}_Category"] = y_smote
            smote_path = os.path.join(OUTPUT_DIR,
                f"{station_id}_{target_name}_SMOTE_balanced.csv")
            smote_df.to_csv(smote_path, index=False)
            print(f"    ✅ {target_name} SMOTE balanced: {smote_path}")

        # Save best hybrid (SMOTE-ENN) if available
        if "4_SMOTE_ENN" in techniques:
            X_se, y_se, _ = techniques["4_SMOTE_ENN"]
            se_df = pd.DataFrame(X_se, columns=feature_cols)
            se_df[f"{target_name}_Category"] = y_se
            se_path = os.path.join(OUTPUT_DIR,
                f"{station_id}_{target_name}_SMOTE_ENN_balanced.csv")
            se_df.to_csv(se_path, index=False)
            print(f"    ✅ {target_name} SMOTE-ENN balanced: {se_path}")

    # Cleanup
    del hi_techniques, pet_techniques
    gc.collect()

    return df


# ============================================================
# RUN
# ============================================================
if __name__ == "__main__":
    print("=" * 70)
    print("  HI & PET COMPUTATION + CLASS BALANCING PIPELINE")
    print("  Mumbai Santacruz (43003) & Colaba (43057)")
    print("=" * 70)

    for station_id, filepath in FILES.items():
        if os.path.exists(filepath):
            df = process_station(station_id, filepath)
            del df
            gc.collect()
        else:
            print(f"\n  ⚠️ File not found: {filepath}")
            print(f"     Please update INPUT_DIR or ensure imputed files are available.")

    print(f"\n{'='*70}")
    print(f"  ALL DONE!")
    print(f"  Output files in: {OUTPUT_DIR}")
    print(f"{'='*70}")


  HI & PET COMPUTATION + CLASS BALANCING PIPELINE
  Mumbai Santacruz (43003) & Colaba (43057)

######################################################################
#  Station 43003
######################################################################

  Loading imputed data...
    Rows: 155390

  Computing Heat Index (HI)...
    HI range: 8.9 to 77.0 °C
    HI mean:  29.8 °C

  Computing PET...
    Estimating Tmrt from air temp, cloud cover, and time of day...
    Computing PET (this may take a few minutes)...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

      PET computation: 26% done...


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

      PET computation: 45% done...


...

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

    ✅ PET SMOTE balanced: /content/drive/MyDrive/Major_project_imd/Class_balancing_output/43057_PET_SMOTE_balanced.csv


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

    ✅ PET SMOTE-ENN balanced: /content/drive/MyDrive/Major_project_imd/Class_balancing_output/43057_PET_SMOTE_ENN_balanced.csv

  ALL DONE!
  Output files in: /content/drive/MyDrive/Major_project_imd/Class_balancing_output


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packag

In [5]:
import pandas as pd

for station in ['43003', '43057']:
    filepath = f"/content/drive/MyDrive/Major_project_imd/Class_balancing_output/{station}_with_HI_PET_categories.csv"
    df = pd.read_csv(filepath, low_memory=False)
    print(f"\n{station} — Zero counts:")
    for col in ['DBT', 'WBT', 'DPT', 'RH', 'VP', 'FFF']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
        n_zero = (df[col] == 0).sum()
        pct = n_zero / len(df) * 100
        print(f"  {col}: {n_zero} zeros ({pct:.2f}%)")


43003 — Zero counts:
  DBT: 0 zeros (0.00%)
  WBT: 0 zeros (0.00%)
  DPT: 2 zeros (0.00%)
  RH: 6 zeros (0.00%)
  VP: 5 zeros (0.00%)
  FFF: 50632 zeros (32.58%)

43057 — Zero counts:
  DBT: 0 zeros (0.00%)
  WBT: 0 zeros (0.00%)
  DPT: 1 zeros (0.00%)
  RH: 1 zeros (0.00%)
  VP: 0 zeros (0.00%)
  FFF: 11589 zeros (19.34%)
